In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np


# =========================
# Config
# =========================
ROOT = Path(r"D:\mmwave-heart-rate-monitoring-demo\results\filter_v1")

FILES = [
    "AWR_steady.csv",
    "AWR_unsteady.csv",
    "IWR_steady.csv",
    "IWR_unsteady.csv",
]

METHOD_ORDER = [
    "raw",
    "AMP",
    "BP",
    "Median",
    "Wavelet",
    "AMP→BP",
    "AMP→Median",
    "AMP→Wavelet",
    "BP→AMP",
    "BP→Median",
    "BP→Wavelet",
    "Median→AMP",
    "Median→BP",
    "Median→Wavelet",
    "Wavelet→AMP",
    "Wavelet→BP",
    "Wavelet→Median",
]

METHOD_TO_COLUMN = {
    "raw": "raw_HR",
    "AMP": "AMP_HR",
    "BP": "BP_HR",
    "Median": "Median_HR",
    "Wavelet": "Wavelet_HR",
    "AMP→BP": "AMP->BP_HR",
    "AMP→Median": "AMP->Median_HR",
    "AMP→Wavelet": "AMP->Wavelet_HR",
    "BP→AMP": "BP->AMP_HR",
    "BP→Median": "BP->Median_HR",
    "BP→Wavelet": "BP->Wavelet_HR",
    "Median→AMP": "Median->AMP_HR",
    "Median→BP": "Median->BP_HR",
    "Median→Wavelet": "Median->Wavelet_HR",
    "Wavelet→AMP": "Wavelet->AMP_HR",
    "Wavelet→BP": "Wavelet->BP_HR",
    "Wavelet→Median": "Wavelet->Median_HR",
}


# =========================
# Metrics
# =========================
def compute_metrics(ecg, mmw):
    error = mmw - ecg
    mae = np.mean(np.abs(error))
    rmse = np.sqrt(np.mean(error**2))
    bias = np.mean(error)
    std = np.std(error, ddof=1) if len(error) > 1 else 0.0
    ci = 1.96 * std
    return mae, rmse, bias, std, ci


# =========================
# Analysis
# =========================
for file in FILES:
    path = ROOT / file

    if not path.exists():
        print(f"找不到檔案: {path}")
        continue

    df = pd.read_csv(path)

    if "ECG_HR" not in df.columns:
        print(f"{file} 缺少 ECG_HR 欄位")
        continue

    ecg = pd.to_numeric(df["ECG_HR"], errors="coerce")
    rows = []

    for method in METHOD_ORDER:
        col = METHOD_TO_COLUMN[method]

        if col not in df.columns:
            rows.append({
                "Method": method,
                "MAE": np.nan,
                "RMSE": np.nan,
                "Bias": np.nan,
                "Std": np.nan,
                "95% CI": np.nan
            })
            continue

        mmw = pd.to_numeric(df[col], errors="coerce")
        valid = np.isfinite(mmw) & np.isfinite(ecg)

        if valid.sum() == 0:
            rows.append({
                "Method": method,
                "MAE": np.nan,
                "RMSE": np.nan,
                "Bias": np.nan,
                "Std": np.nan,
                "95% CI": np.nan
            })
            continue

        mae, rmse, bias, std, ci = compute_metrics(
            ecg[valid].to_numpy(),
            mmw[valid].to_numpy()
        )

        rows.append({
            "Method": method,
            "MAE": round(mae, 3),
            "RMSE": round(rmse, 3),
            "Bias": round(bias, 3),
            "Std": round(std, 3),
            "95% CI": round(ci, 3)
        })

    result = pd.DataFrame(rows)

    print("\n" + "=" * 60)
    print(file)
    print("=" * 60)
    print(result.to_string(index=False))

    # out_path = ROOT / f"{Path(file).stem}_stats.csv"
    # result.to_csv(out_path, index=False, encoding="utf-8-sig")
    # print(f"\n已儲存: {out_path}")


AWR_steady.csv
        Method    MAE   RMSE   Bias    Std  95% CI
           raw  5.657  7.511  2.312  7.219  14.150
           AMP  5.667  7.385  2.462  7.034  13.786
            BP  5.647  7.506  2.462  7.163  14.040
        Median 17.290 21.402  9.775 19.232  37.695
       Wavelet 21.207 31.941 13.037 29.455  57.732
        AMP→BP  5.817  7.461  2.612  7.060  13.838
    AMP→Median 16.915 20.934  9.400 18.895  37.034
   AMP→Wavelet 15.712 23.594  0.023 23.834  46.714
        BP→AMP  5.817  7.461  2.612  7.060  13.838
     BP→Median 16.748 20.964 13.338 16.338  32.022
    BP→Wavelet 22.929 35.043 16.224 31.376  61.498
    Median→AMP 16.430 19.583  7.900 18.101  35.477
     Median→BP 17.135 21.276 16.075 14.079  27.595
Median→Wavelet 17.077 23.445  6.662 22.707  44.505
   Wavelet→AMP 11.539 14.104 -0.276 14.244  27.919
    Wavelet→BP 19.729 32.214 15.699 28.415  55.693
Wavelet→Median 16.240 17.733 -3.425 17.575  34.448

已儲存: D:\mmwave-heart-rate-monitoring-demo\results\filter_v1\AWR_s